# SIGMOD Exp2 Distinct SNAP Views

Rerun the distinct crossover sweeps from scratch and render only the SNAP-included plain and broken-axis views for history and delta sweeps.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import importlib

ROOT = Path('../../').resolve()
sys.path.append(str(ROOT / 'benches'))
import sigmod_exp_common as _sigmod_exp_common
importlib.reload(_sigmod_exp_common)
sys.path.append(str(ROOT / 'benches' / 'hash_join' / 'htap_simulation'))

from sigmod_exp_common import (
    TOL,
    SIGMOD_BUCKET_NUM,
    SIGMOD_HTAP_TXN_COUNT,
    SIGMOD_HTAP_WAREHOUSE_COUNT,
    SIGMOD_READABLE_EVERY,
    apply_paper_style,
    current_run_stamp,
    ensure_dirs,
    run_checked,
)
from bench_script_functions import parse_result

apply_paper_style(ROOT)

EXP_DIR = (ROOT / 'benches' / 'sigmod_exp2_distinct_snap_views').resolve()
DATA_DIR = EXP_DIR / 'data'
FIGS_DIR = EXP_DIR / 'figs'
ensure_dirs(DATA_DIR, FIGS_DIR)

BIN = ROOT / 'target' / 'release' / 'htap_wkld'
TABLE_TYPES = ['naive', 'ivmh', 'heap', 'chain', 'par']
TX_MAP = {'MarkTs': 'BuildSnap', 'DelSc': 'DeltaScan'}

CONFIG = {
    'warehouse_count': SIGMOD_HTAP_WAREHOUSE_COUNT,
    'txn_count': SIGMOD_HTAP_TXN_COUNT,
    'bucket_num': SIGMOD_BUCKET_NUM,
    'update_ratio': 0.0001,
    'probe_ratio': 0.01,
    'txn_gc_ratio': 0.05,
    'readable_every': SIGMOD_READABLE_EVERY,
    'repeat': 5,
    'warmup_runs': 1,
    'trim': 1,
    'timeout_sec': 900,
}

SWEEP = {
    'history_values': [0.00, 0.01, 0.02, 0.04, 0.06, 0.08, 0.10],
    'delta_values': [0.00, 0.01, 0.02, 0.04, 0.06, 0.08, 0.10],
}

STYLE = {
    ('naive', ''): ('SNAP', TOL['red'], ':', 'x'),
    ('ivmh', ''): ('IVMH', TOL['yellow'], '--', 'P'),
    ('heap', 'Write Repair'): ('MONO-WR', TOL['blue'], '-', 'o'),
    ('chain', 'Write Repair'): ('DUAL-WR', TOL['cyan'], '-', 's'),
    ('par', 'Write Repair'): ('EPOCH-WR', TOL['green'], '-', 'D'),
}

RUN_STAMP = current_run_stamp()
RUN_TAG = '_'.join([
    f"wc{CONFIG['warehouse_count']}",
    f"tc{CONFIG['txn_count']}",
    f"bn{CONFIG['bucket_num']}",
    f"ur{str(CONFIG['update_ratio']).replace('.', 'p')}",
    f"pr{str(CONFIG['probe_ratio']).replace('.', 'p')}",
    f"gc{str(CONFIG['txn_gc_ratio']).replace('.', 'p')}",
    f"re{CONFIG['readable_every']}",
    f"rep{CONFIG['repeat']}",
    'distinct',
    RUN_STAMP,
])

BASE_ARGS = [
    '--txn-count', str(CONFIG['txn_count']),
    '--warehouse-count', str(CONFIG['warehouse_count']),
    '--update-ratio', str(CONFIG['update_ratio']),
    '--probe-ratio', str(CONFIG['probe_ratio']),
    '--bucket-num', str(CONFIG['bucket_num']),
    '--readable-every', str(CONFIG['readable_every']),
]

print('ROOT   :', ROOT)
print('BIN    :', BIN)
print('OUTDIR :', DATA_DIR)
print('CONFIG :', CONFIG)
print('STAMP  :', RUN_STAMP)
print('TAG    :', RUN_TAG)


In [ ]:
print('Building htap_wkld...')
run_checked(['cargo', 'build', '--release', '--bin', 'htap_wkld'], ROOT)
print('Build OK')


In [ ]:
def merge_args(base, extra):
    return [*base, *extra]


def trim_trial_runs(df):
    trim = CONFIG['trim']
    if trim <= 0 or df.empty or 'trial' not in df.columns:
        return df
    totals = (
        df.groupby('trial', as_index=False)['duration_ms']
        .sum()
        .sort_values('duration_ms')
    )
    if len(totals) <= 2 * trim:
        return df
    keep = set(totals.iloc[trim:len(totals) - trim]['trial'])
    return df[df['trial'].isin(keep)].copy()


def collapse_repairs(df, table_type):
    if table_type in {'naive', 'ivmh'}:
        collapsed = df.groupby(['table_type'], as_index=False)['duration_ms'].mean()
        collapsed['repair_type'] = ''
        return collapsed[['table_type', 'repair_type', 'duration_ms']]
    return df


def run_single_table(extra_args, table_type):
    trials = []
    print('  table=', table_type)
    for warmup in range(CONFIG['warmup_runs']):
        print(f'    warmup {warmup + 1}/{CONFIG["warmup_runs"]}')
        _ = run_checked(
            [str(BIN), *merge_args(BASE_ARGS, extra_args), '--table-type', table_type],
            ROOT,
            quiet=True,
            timeout=CONFIG['timeout_sec'],
        )
    for trial in range(CONFIG['repeat']):
        result = run_checked(
            [str(BIN), *merge_args(BASE_ARGS, extra_args), '--table-type', table_type],
            ROOT,
            quiet=True,
            timeout=CONFIG['timeout_sec'],
        )
        df = parse_result(result.stdout, table_type)
        if df.empty:
            raise RuntimeError(f'No parsed rows for {table_type}')
        df['tx_type'] = df['tx_type'].replace(TX_MAP)
        total = df.groupby('repair_type', as_index=False)['duration_ms'].sum()
        total['table_type'] = table_type
        total['trial'] = trial
        trials.append(total)
    df_all = trim_trial_runs(pd.concat(trials, ignore_index=True))
    df_avg = df_all.groupby(['table_type', 'repair_type'], as_index=False)['duration_ms'].mean()
    out = collapse_repairs(df_avg, table_type)
    out['total_ms'] = out['duration_ms'] / CONFIG['txn_count']
    return out


def effective_tx_counts(update_ratio, probe_ratio, scan_ratio, delta_ratio):
    gc_ratio = CONFIG['txn_gc_ratio']
    analytical_and_update = 1.0 - gc_ratio
    update_n = round(CONFIG['txn_count'] * analytical_and_update * update_ratio)
    probe_n = round(CONFIG['txn_count'] * analytical_and_update * probe_ratio)
    delta_n = round(CONFIG['txn_count'] * analytical_and_update * delta_ratio)
    gc_n = round(CONFIG['txn_count'] * gc_ratio)
    scan_n = CONFIG['txn_count'] - update_n - probe_n - delta_n - gc_n
    return update_n, probe_n, scan_n, delta_n, gc_n


def build_history_args(history_pct):
    total_history_ops = int(round(CONFIG['txn_count'] * history_pct))
    update_ratio = 0.20
    probe_ratio = 0.40
    delta_ratio = 0.00
    scan_ratio = 0.40
    _, actual_probe_count, actual_scan_count, _, _ = effective_tx_counts(update_ratio, probe_ratio, scan_ratio, delta_ratio)
    if total_history_ops <= 1:
        history_scan_count = total_history_ops
        history_probe_count = 0
    else:
        history_probe_count = total_history_ops // 2
        history_scan_count = total_history_ops - history_probe_count
    scan_reuse_ratio = 0.0 if actual_scan_count == 0 else min(1.0, history_scan_count / actual_scan_count)
    probe_history_ratio = 0.0 if actual_probe_count == 0 else min(1.0, history_probe_count / actual_probe_count)
    return [
        '--txn-update-ratio', str(update_ratio),
        '--txn-probe-ratio', str(probe_ratio),
        '--txn-scan-ratio', str(scan_ratio),
        '--txn-delta-ratio', str(delta_ratio),
        '--txn-gc-ratio', str(CONFIG['txn_gc_ratio']),
        '--scan-reuse-ratio', str(scan_reuse_ratio),
        '--probe-history-ratio', str(probe_history_ratio),
        '--distinct-history-targets',
    ]


def build_delta_args(delta_pct):
    delta_ratio = float(delta_pct)
    update_ratio = 0.20
    scan_ratio = 0.40
    probe_ratio = 1.0 - update_ratio - scan_ratio - delta_ratio
    if probe_ratio < 0:
        raise ValueError(f'Invalid delta_pct {delta_pct}: negative probe share')
    return [
        '--txn-update-ratio', str(update_ratio),
        '--txn-probe-ratio', str(probe_ratio),
        '--txn-scan-ratio', str(scan_ratio),
        '--txn-delta-ratio', str(delta_ratio),
        '--txn-gc-ratio', str(CONFIG['txn_gc_ratio']),
        '--scan-reuse-ratio', '0.0',
        '--probe-history-ratio', '0.0',
        '--distinct-delta-targets',
    ]


def run_sweep(x_col, values, make_args_fn, stem):
    rows = []
    for value in values:
        print(f'Running {x_col}={value:.2%}')
        extra_args = make_args_fn(value)
        for table_type in TABLE_TYPES:
            table_df = run_single_table(extra_args, table_type)
            table_df[x_col] = float(value)
            rows.append(table_df)
    out = pd.concat(rows, ignore_index=True)
    out = out.sort_values([x_col, 'table_type', 'repair_type']).reset_index(drop=True)
    csv_path = DATA_DIR / f'{stem}_{RUN_TAG}.csv'
    out.to_csv(csv_path, index=False)
    print('Saved', csv_path)
    return out, csv_path


In [ ]:
def plot_one(ax, df, x_col, xlabel):
    series = [
        ('naive', ''),
        ('ivmh', ''),
        ('heap', 'Write Repair'),
        ('chain', 'Write Repair'),
        ('par', 'Write Repair'),
    ]
    for key in series:
        label, color, linestyle, marker = STYLE[key]
        table_type, repair_type = key
        sub = df[(df['table_type'] == table_type) & (df['repair_type'] == repair_type)].sort_values(x_col)
        if sub.empty:
            continue
        ax.plot(sub[x_col], sub['total_ms'], color=color, linestyle=linestyle, marker=marker, linewidth=1.8, markersize=5, label=label)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Duration (ms / tx)')
    ax.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
    ax.set_ylim(bottom=0)
    ax.legend(loc='center left', ncol=1, framealpha=0.95)


def snap_break_bounds(df):
    snap = df[df['table_type'] == 'naive']['total_ms']
    others = df[df['table_type'] != 'naive']['total_ms']
    if snap.empty or others.empty:
        return None
    lower_max = others.max() * 1.08
    upper_min = snap.min() * 0.92
    if upper_min <= lower_max:
        midpoint = (others.max() + snap.min()) / 2.0
        lower_max = midpoint * 0.95
        upper_min = midpoint * 1.05
    upper_max = snap.max() * 1.03
    return lower_max, upper_min, upper_max


def save_fig(fig, stem):
    stamped = FIGS_DIR / f'{stem}_{RUN_STAMP}.pdf'
    latest = FIGS_DIR / f'{stem}.pdf'
    fig.savefig(stamped, format='pdf', bbox_inches='tight')
    fig.savefig(latest, format='pdf', bbox_inches='tight')
    plt.show()
    print('Saved', stamped)
    print('Saved', latest)


def render_plain(df, x_col, xlabel, stem):
    fig, ax = plt.subplots(1, 1, figsize=(5.0, 4.1))
    plot_one(ax, df, x_col, xlabel)
    fig.tight_layout()
    save_fig(fig, stem)


def render_broken(df, x_col, xlabel, stem):
    bounds = snap_break_bounds(df)
    if bounds is None:
        render_plain(df, x_col, xlabel, stem)
        return
    lower_max, upper_min, upper_max = bounds
    fig, (ax_top, ax_bottom) = plt.subplots(
        2, 1,
        figsize=(5.0, 4.6),
        sharex=True,
        gridspec_kw={'height_ratios': [1.0, 3.0], 'hspace': 0.05},
    )
    plot_one(ax_top, df, x_col, '')
    plot_one(ax_bottom, df, x_col, xlabel)
    ax_top.set_ylim(upper_min, upper_max)
    ax_bottom.set_ylim(0, lower_max)
    ax_top.spines['bottom'].set_visible(False)
    ax_bottom.spines['top'].set_visible(False)
    ax_top.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
    ax_top.set_xlabel('')
    ax_top.set_ylabel('')
    if ax_bottom.legend_ is not None:
        ax_bottom.legend_.remove()
    d = 0.012
    kwargs = dict(transform=ax_top.transAxes, color='k', clip_on=False, linewidth=0.8)
    ax_top.plot((-d, +d), (-d, +d), **kwargs)
    ax_top.plot((1 - d, 1 + d), (-d, +d), **kwargs)
    kwargs.update(transform=ax_bottom.transAxes)
    ax_bottom.plot((-d, +d), (1 - d, 1 + d), **kwargs)
    ax_bottom.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)
    fig.tight_layout()
    save_fig(fig, stem)


In [ ]:
df_history, history_csv = run_sweep(
    'history_ratio',
    SWEEP['history_values'],
    build_history_args,
    'sigmod_exp2_distinct_history_snap_view',
)
df_history['history_pct'] = df_history['history_ratio'] * 100.0
print('Saved', history_csv)
display(df_history.head())
render_plain(df_history, 'history_pct', 'Historical Scan Percentage (%)', 'exp2-distinct-history-with-snap-view')
render_broken(df_history, 'history_pct', 'Historical Scan Percentage (%)', 'exp2-distinct-history-with-snap-broken-view')


In [ ]:
df_delta, delta_csv = run_sweep(
    'delta_ratio',
    SWEEP['delta_values'],
    build_delta_args,
    'sigmod_exp2_distinct_delta_snap_view',
)
df_delta['delta_pct'] = df_delta['delta_ratio'] * 100.0
print('Saved', delta_csv)
display(df_delta.head())
render_plain(df_delta, 'delta_pct', 'Delta Transaction Percentage (%)', 'exp2-distinct-delta-with-snap-view')
render_broken(df_delta, 'delta_pct', 'Delta Transaction Percentage (%)', 'exp2-distinct-delta-with-snap-broken-view')
